In [ ]:
from hydra import initialize, compose
from omegaconf import DictConfig
import os

from src.data.dataloader import data_loader
from src.data.dataset_aprdb import AprdbDataset
from src.detector.detector_factory import detector_factory
from src.evaluation.eval_engine import Evaluator
import src.config
with initialize(version_base=None, config_path="../conf"):
    cfg: DictConfig = compose(config_name="config")

os.environ['HF_HOME'] = "/data2/cyaiche/hf_cache"
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

print(cfg)

In [ ]:
ds_test = AprdbDataset(config=cfg.experiment.dataset)

In [ ]:
dataloader = data_loader(
    cfg.experiment.data_loader,
    ds_test,
)

In [ ]:
model = detector_factory(cfg.experiment.detector)

In [ ]:
bino = model.binocular

In [ ]:

print("tokenizer.vocab_size     :", bino.tokenizer.vocab_size)
print("len(tokenizer)           :", len(bino.tokenizer))  # inclut les special tokens ajoutés
print("observer config.vocab_size :", bino.observer_model.config.vocab_size)
print("performer config.vocab_size:", bino.performer_model.config.vocab_size)

In [ ]:
print(cfg.experiment.device)
evaluator = Evaluator(
    model=model,
    dataloader=dataloader,
    device=cfg.experiment.device,
    type=cfg.experiment.detector.type,
)

In [ ]:
for batch in evaluator.dataloader:
    texts = batch['content']
    enc = evaluator.model.binocular._tokenize(texts)
    max_id = enc['input_ids'].max().item()
    min_id = enc['input_ids'].min().item()
    if max_id >= 65024 or min_id < 0:
        print(f"BATCH FAUTIF — max_id: {max_id}, min_id: {min_id}")
        print(f"Textes: {texts}")
        break
    #print(f"OK — max_id: {max_id}")


In [ ]:
print("pad_token_id:", bino.tokenizer.pad_token_id)
print("eos_token_id:", bino.tokenizer.eos_token_id)

In [ ]:
loader_iter = iter(evaluator.dataloader)
next(loader_iter)  # skip batch 0
batch1 = next(loader_iter)

texts = batch1['content']
print("Texte:", texts)

enc = bino._tokenize(texts)
print("IDs — min:", enc['input_ids'].min().item(), "max:", enc['input_ids'].max().item())

# Observer seul
obs = bino.observer_model(**enc.to("cuda:0")).logits
print("Observer OK")

# Performer seul
perf = bino.performer_model(**enc.to("cuda:1")).logits
print("Performer OK")

In [ ]:
bino = evaluator.model.binocular
import torch

for i, batch in enumerate(evaluator.dataloader):
    texts = batch['content']
    enc = bino._tokenize(texts)
    try:
        obs = bino.observer_model(**enc.to("cuda:0")).logits
        perf = bino.performer_model(**enc.to("cuda:1")).logits
        torch.cuda.synchronize()
        print(f"Batch {i} OK")
    except Exception as e:
        print(f"Batch {i} FAUTIF — shape: {enc['input_ids'].shape}")
        print(f"Erreur: {e}")
        break

In [ ]:
bino = evaluator.model.binocular

for i, batch in enumerate(evaluator.dataloader):
    texts = batch['content']
    # Tokenisation CPU pure — pas de .to(device)
    enc = bino.tokenizer(
        texts,
        return_tensors="pt",
        padding="longest" if len(texts) > 1 else False,
        truncation=True,
        max_length=bino.max_token_observed,
        return_token_type_ids=False,
    )
    max_id = enc['input_ids'].max().item()
    min_id = enc['input_ids'].min().item()
    if max_id >= 65024 or min_id < 0:
        print(f"Batch {i} FAUTIF — max: {max_id}, min: {min_id}")
        print(texts)
        break
    print(f"Batch {i} OK — max_id: {max_id}, shape: {enc['input_ids'].shape}")

In [ ]:
bino = evaluator.model.binocular
print(bino.observer_model.transformer.word_embeddings.weight.shape)
print(bino.performer_model.transformer.word_embeddings.weight.shape)

In [ ]:
batch = next(iter(evaluator.dataloader))
texts = batch['content']

bino = evaluator.model.binocular
enc = bino._tokenize(texts)
print("input_ids — min:", enc['input_ids'].min().item(), "max:", enc['input_ids'].max().item())

# Observer seul
obs = bino.observer_model(**enc.to("cuda:0")).logits
print("Observer OK — shape:", obs.shape)

# Performer seul
perf = bino.performer_model(**enc.to("cuda:1")).logits
print("Performer OK — shape:", perf.shape)

In [ ]:
bino = evaluator.model.binocular

for i, batch in enumerate(evaluator.dataloader):
    texts = batch['content']
    try:
        enc = bino._tokenize(texts)
        obs = bino.observer_model(**enc.to("cuda:0")).logits
        perf = bino.performer_model(**enc.to("cuda:1")).logits
        print(f"Batch {i} OK — seq_len: {enc['input_ids'].shape[1]}")
    except Exception as e:
        print(f"Batch {i} FAUTIF")
        print(f"input_ids min/max: {enc['input_ids'].min().item()} / {enc['input_ids'].max().item()}")
        print(f"texts: {texts}")
        print(f"Erreur: {e}")
        break

In [ ]:
#metrics = evaluator.run()